# Kinship Library

## Books for Living in Connection

This notebook develops the project from Day 1 through Day 7.

The recommender focuses on five connected reading areas.

1. Humans and nature
2. Yoga philosophy
3. Yoga as a way of life
4. Ayurveda and Indian spirituality
5. Indigenous knowledge from South America, with specific search attention to Kogi and Muisca sources

The project does not assume that these traditions are interchangeable. The categories are used as discovery paths. Book metadata is kept as close as possible to the language provided by the original data sources.

## Main question

Can book descriptions and subject metadata help readers discover meaningful books about living in connection with self, community, nature, and ancestral knowledge?

## Analytical path

Business understanding  
Data mining  
Data cleaning  
Data exploration  
Feature engineering  
Predictive modeling through unsupervised learning and similarity  
Data visualization  
SQL storage  
API enrichment

## Day 0. Project setup

### Purpose

Create a reproducible environment and use the same folder and variable names throughout the project.

### Expected result

The imports should run without errors. The project paths should point to folders inside this repository.

In [2]:
from pathlib import Path
import re
import time
import sqlite3
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
DATABASE_PATH = PROJECT_ROOT / "data" / "kinship_library.db"

RAW_DATA.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# Day 1. Business understanding and data collection

## Business understanding

### Problem

Readers interested in ecology, Yoga, Ayurveda, Indian spirituality, and Indigenous knowledge can find books across many catalogues, but discovery is often fragmented by genre labels and commercial categories.

### Objective

Build a content based recommender that uses book descriptions and subjects to suggest related reading while allowing users to explore distinct knowledge areas.

### Questions

1. Which themes are most represented in the collected catalogue?
2. Which authors and subjects appear most often?
3. Are descriptions complete enough for NLP?
4. Do textual features reveal meaningful clusters?
5. Does a book receive sensible recommendations from cosine similarity?
6. Can external APIs improve incomplete metadata?

### Success criteria

The final dataset should contain enough complete descriptions for TF IDF modeling.

The model should produce interpretable clusters.

Recommendations should have a visible similarity score and enough metadata for a reader to understand the result.

The complete workflow should run from repository relative paths.

## Data strategy

Books to Scrape is used to demonstrate web scraping safely.

Open Library is used for thematic discovery.

Google Books is used as a second API source and later as enrichment.

The search vocabulary is intentionally broad. This helps us collect books from related areas without treating all knowledge traditions as one category.

In [3]:
THEME_QUERIES = {
    "humans_and_nature": [
        "ecology human nature",
        "environmental humanities",
        "more than human",
        "ecofeminism",
        "deep ecology",
        "indigenous ecology"
    ],
    "yoga_philosophy": [
        "Yoga Sutras",
        "Vedas",
        "Upanishads",
        "Bhagavad Gita",
        "yoga philosophy"
    ],
    "yoga_way_of_life": [
        "yoga practice philosophy",
        "eight limbs yoga",
        "pranayama",
        "meditation yoga"
    ],
    "ayurveda_indian_spirituality": [
        "Ayurveda",
        "Indian spirituality",
        "Vedanta",
        "Tantra India",
        "Autobiography of a Yogi"
    ],
    "south_american_indigenous_knowledge": [
        "Kogi Colombia",
        "Muisca Colombia",
        "Andean indigenous knowledge",
        "South America indigenous ecology",
        "Colombia indigenous cosmology"
    ]
}

SEED_TITLES = [
    "Staying with the Trouble",
    "Ancestral Future",
    "The Yoga Sutras of Patanjali",
    "Autobiography of a Yogi",
    "Bhagavad Gita"
]

## 1A. Scrape Books to Scrape

### Purpose

Demonstrate ethical beginner level scraping and collect a general comparison sample.

### Code

The loop reads catalogue pages, extracts visible metadata, and pauses between requests.

### Expected result

A dataframe called `scraped_books` with title, price, rating, availability, source, and source category.

In [ ]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

scraped_rows = []

for page in range(1, 11):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.select("article.product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.select_one(".price_color").get_text(strip=True)
        availability = book.select_one(".availability").get_text(strip=True)
        rating_word = book.p.get("class", ["", ""])[1]

        scraped_rows.append({
            "title": title,
            "author": np.nan,
            "description": np.nan,
            "subjects": np.nan,
            "published_year": np.nan,
            "average_rating": rating_map.get(rating_word),
            "ratings_count": np.nan,
            "isbn": np.nan,
            "language": "en",
            "source": "books_to_scrape",
            "source_category": "general_catalogue",
            "price": price,
            "availability": availability
        })

    time.sleep(1)

scraped_books = pd.DataFrame(scraped_rows)
scraped_books.to_csv(RAW_DATA / "books_to_scrape.csv", index=False)

scraped_books.head()

,title,author,description,subjects,published_year,average_rating,ratings_count,isbn,language,source,source_category,price,availability
0,A Light in the Attic,NaN,NaN,NaN,NaN,3,NaN,NaN,en,books_to_scrape,general_catalogue,Â£51.77,In stock
1,Tipping the Velvet,NaN,NaN,NaN,NaN,1,NaN,NaN,en,books_to_scrape,general_catalogue,Â£53.74,In stock
2,Soumission,NaN,NaN,NaN,NaN,1,NaN,NaN,en,books_to_scrape,general_catalogue,Â£50.10,In stock
3,Sharp Objects,NaN,NaN,NaN,NaN,4,NaN,NaN,en,books_to_scrape,general_catalogue,Â£47.82,In stock
4,Sapiens: A Brief History of Humankind,NaN,NaN,NaN,NaN,5,NaN,NaN,en,books_to_scrape,general_catalogue,Â£54.23,In stock


## 1B. Collect thematic books from Open Library

### Purpose

Collect books that are closer to the subject of Kinship Library.

### Expected result

A dataframe called `open_library_books` containing book level metadata and the project theme that led to each result.

In [ ]:
open_library_rows = []

headers = {
    "User-Agent": "KinshipLibraryStudentProject contact@example.com"
}

for project_theme, queries in THEME_QUERIES.items():
    for query in queries:
        params = {
            "q": query,
            "limit": 25,
            "fields": "key,title,author_name,first_publish_year,subject,isbn,language,ratings_average,ratings_count"
        }

        response = requests.get(
            "https://openlibrary.org/search.json",
            params=params,
            headers=headers,
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        for book in data.get("docs", []):
            open_library_rows.append({
                "open_library_key": book.get("key"),
                "title": book.get("title"),
                "author": ", ".join(book.get("author_name", [])),
                "description": np.nan,
                "subjects": ", ".join(book.get("subject", [])[:10]),
                "published_year": book.get("first_publish_year"),
                "average_rating": book.get("ratings_average"),
                "ratings_count": book.get("ratings_count"),
                "isbn": book.get("isbn", [np.nan])[0] if book.get("isbn") else np.nan,
                "language": book.get("language", [np.nan])[0] if book.get("language") else np.nan,
                "source": "open_library",
                "source_category": project_theme,
                "search_query": query
            })

        time.sleep(1)

open_library_books = pd.DataFrame(open_library_rows)
open_library_books.to_csv(RAW_DATA / "open_library_books.csv", index=False)

open_library_books.head()

,open_library_key,title,author,description,subjects,published_year,average_rating,ratings_count,isbn,language,source,source_category,search_query
0,/works/OL6772557W,The ecology of human development,Urie Bronfenbrenner,NaN,"Child psychology, Research, Human Development,...",1979.0,5.000000,1.0,9780674224575,eng,open_library,humans_and_nature,ecology human nature
1,/works/OL3458484W,People and nature,Emilio F. Moran,NaN,"Effect of environment on, Effect of environmen...",2006.0,NaN,NaN,1118877411,eng,open_library,humans_and_nature,ecology human nature
2,/works/OL2990533W,The green imperative,"Victor J. Papanek, Victor Papanek",NaN,"Design, industrial, Environmental aspects, Env...",1995.0,NaN,NaN,9780500296196,eng,open_library,humans_and_nature,ecology human nature
3,/works/OL1805267W,Design with nature,Ian L. McHarg,NaN,"Effect of environment on, Effect of human bein...",1969.0,4.333334,6.0,047111460X,eng,open_library,humans_and_nature,ecology human nature
4,/works/OL2911248W,"The death of nature: women, ecology, and the s...",Carolyn Merchant,NaN,"Human ecology, Philosophy of nature, Social as...",1980.0,NaN,NaN,9780062505712,eng,open_library,humans_and_nature,ecology human nature


In [8]:
import requests

url = "https://www.googleapis.com/books/v1/volumes"

params = {
    "q": "harry potter",
    "maxResults": 1
}

response = requests.get(url, params=params)

print("Status:", response.status_code)
print("Response:")
print(response.text[:1000])

Status: 429
Response:
{
  "error": {
    "code": 429,
    "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'books.googleapis.com' for consumer 'project_number:624717413613'.",
    "errors": [
      {
        "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'books.googleapis.com' for consumer 'project_number:624717413613'.",
        "domain": "global",
        "reason": "rateLimitExceeded"
      }
    ],
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "RATE_LIMIT_EXCEEDED",
        "domain": "googleapis.com",
        "metadata": {
          "quota_location": "global",
          "quota_limit_value": "0",
          "service": "books.googleapis.com",
          "quota_limit": "defaultPerDayPerProject",
          "quota_metric": "books.googleapis.com/default",
          "quota_unit": "1/d/{project}",
          "con

In [9]:
from google.colab import userdata

GOOGLE_BOOKS_API_KEY = userdata.get(
    "API_key_books"
)

print("API key loaded successfully")

API key loaded successfully


In [10]:
import requests

url = "https://www.googleapis.com/books/v1/volumes"

params = {
    "q": "Staying with the Trouble",
    "maxResults": 1,
    "key": GOOGLE_BOOKS_API_KEY
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print(response.text[:500])

Status: 200
{
  "kind": "books#volumes",
  "totalItems": 300,
  "items": [
    {
      "kind": "books#volume",
      "id": "ZvDgDAAAQBAJ",
      "etag": "yNbPaeQuYaQ",
      "selfLink": "https://www.googleapis.com/books/v1/volumes/ZvDgDAAAQBAJ",
      "volumeInfo": {
        "title": "Staying with the Trouble",
        "subtitle": "Making Kin in the Chthulucene",
        "authors": [
          "Donna J. Haraway"
        ],
        "publisher": "Duke University Press",
        "publishedDate": "2016-08-25",



## 1C. Google Books collection

### Purpose

Add a second API source with descriptions and categories, which are especially useful for NLP.

### Expected result

A dataframe called `google_books` with descriptions, categories, identifiers, ratings, and project themes.

In [7]:
import time
import requests
import pandas as pd
import numpy as np

google_rows = []

for project_theme, queries in THEME_QUERIES.items():
    for query in queries:
        params = {
            "q": query,
            "maxResults": 20,
            "printType": "books"
            }
# Try the request up to 5 times
for attempt in range(5):

            response = requests.get(
                "https://www.googleapis.com/books/v1/volumes",
                params=params,
                timeout=30
            )

            if response.status_code == 200:
                break

            elif response.status_code == 429:
                wait_time = 30 * (attempt + 1)

                print(
                    f"Rate limited for '{query}'. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                response.raise_for_status()

        else:
            print(f"Skipping '{query}' after repeated rate limits.")
            continue

        data = response.json()

        for item in data.get("items", []):

            info = item.get("volumeInfo", {})
            identifiers = info.get("industryIdentifiers", [])

            isbn = (
                identifiers[0].get("identifier")
                if identifiers
                else np.nan
            )

            google_rows.append({
                "google_books_id": item.get("id"),
                "title": info.get("title"),
                "author": ", ".join(info.get("authors", [])),
                "description": info.get("description"),
                "subjects": ", ".join(info.get("categories", [])),
                "published_year": str(
                    info.get("publishedDate", "")
                )[:4],
                "average_rating": info.get("averageRating"),
                "ratings_count": info.get("ratingsCount"),
                "isbn": isbn,
                "language": info.get("language"),
                "source": "google_books",
                "source_category": project_theme,
                "search_query": query
            })

        # Pause before the next query
        time.sleep(10)



google_books = pd.DataFrame(google_rows)

google_books.to_csv(
    RAW_DATA / "google_books.csv",
    index=False
)

SyntaxError: invalid non-printable character U+00A0 (1273334495.py, line 9)

In [4]:
google_rows = []

for project_theme, queries in THEME_QUERIES.items():
    for query in queries:
        params = {
            "q": query,
            "maxResults": 20,
            "printType": "books"
        }

        response = requests.get(
            "https://www.googleapis.com/books/v1/volumes",
            params=params,
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        for item in data.get("items", []):
            info = item.get("volumeInfo", {})
            identifiers = info.get("industryIdentifiers", [])
            isbn = identifiers[0].get("identifier") if identifiers else np.nan

            google_rows.append({
                "google_books_id": item.get("id"),
                "title": info.get("title"),
                "author": ", ".join(info.get("authors", [])),
                "description": info.get("description"),
                "subjects": ", ".join(info.get("categories", [])),
                "published_year": str(info.get("publishedDate", ""))[:4],
                "average_rating": info.get("averageRating"),
                "ratings_count": info.get("ratingsCount"),
                "isbn": isbn,
                "language": info.get("language"),
                "source": "google_books",
                "source_category": project_theme,
                "search_query": query
            })

        time.sleep(10)

google_books = pd.DataFrame(google_rows)
google_books.to_csv(RAW_DATA / "google_books.csv", index=False)

google_books.head()

HTTPError: 429 Client Error: Too Many Requests for url: https://www.googleapis.com/books/v1/volumes?q=ecology+human+nature&maxResults=20&printType=books

In [5]:
import requests

test_url = "https://www.googleapis.com/books/v1/volumes"

test_params = {
    "q": "Staying with the Trouble",
    "maxResults": 5
}

test_response = requests.get(
    test_url,
    params=test_params,
    timeout=30
)

print("Status code:", test_response.status_code)
print(test_response.text[:500])

Status code: 429
{
  "error": {
    "code": 429,
    "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'books.googleapis.com' for consumer 'project_number:624717413613'.",
    "errors": [
      {
        "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'books.googleapis.com' for consumer 'project_number:624717413613'.",
        "domain": "global",
        "reason": "rateLimitExceeded"
      }
    ],
    "status": "RESOURCE_EXH


# Day 2. Data cleaning and exploratory data analysis

## Purpose

Combine the thematic API data, remove duplicate records, standardize fields, inspect missing values, examine distributions, check outliers, and test selected hypotheses.

Books to Scrape remains a separate scraping demonstration because its catalogue lacks the descriptions and thematic metadata needed for the main NLP model.

In [ ]:
books_raw = pd.concat(
    [open_library_books, google_books],
    ignore_index=True,
    sort=False
)

books_raw.shape

In [ ]:
books = books_raw.copy()

books["title"] = books["title"].astype("string").str.strip()
books["author"] = books["author"].astype("string").str.strip()
books["description"] = books["description"].astype("string").str.strip()
books["subjects"] = books["subjects"].astype("string").str.strip()

books["published_year"] = pd.to_numeric(
    books["published_year"],
    errors="coerce"
)

books["average_rating"] = pd.to_numeric(
    books["average_rating"],
    errors="coerce"
)

books["ratings_count"] = pd.to_numeric(
    books["ratings_count"],
    errors="coerce"
)

books = books.dropna(subset=["title"])
books = books.drop_duplicates(
    subset=["title", "author"],
    keep="first"
)

books = books.reset_index(drop=True)

books.info()

## Missing values

The next table shows how much information is missing from every column. Description completeness is especially important because descriptions will become the main NLP text.

In [ ]:
missing_values = pd.DataFrame({
    "missing_count": books.isnull().sum(),
    "missing_percent": books.isnull().mean() * 100
}).sort_values("missing_percent", ascending=False)

missing_values

In [ ]:
books["description_length"] = (
    books["description"]
    .fillna("")
    .str.split()
    .str.len()
)

books["subject_count"] = (
    books["subjects"]
    .fillna("")
    .apply(lambda text: 0 if text == "" else len(text.split(",")))
)

books.describe(include="all")

## Univariate exploration

We inspect one variable at a time.

The rating distribution shows whether ratings are concentrated around a narrow range.

The description length distribution shows whether a small number of very long descriptions create outliers.

In [ ]:
plt.figure(figsize=(8, 5))
books["average_rating"].dropna().hist(bins=15)
plt.title("Distribution of average book ratings")
plt.xlabel("Average rating")
plt.ylabel("Number of books")
plt.show()

plt.figure(figsize=(8, 5))
books["description_length"].hist(bins=30)
plt.title("Distribution of description length")
plt.xlabel("Words in description")
plt.ylabel("Number of books")
plt.show()

## Outlier check with IQR

### Purpose

Identify unusually long descriptions without automatically deleting them.

An outlier is not necessarily an error. We inspect it before deciding whether it should be removed.

In [ ]:
q1 = books["description_length"].quantile(0.25)
q3 = books["description_length"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

description_outliers = books[
    (books["description_length"] < lower_bound) |
    (books["description_length"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(description_outliers))

description_outliers[
    ["title", "author", "description_length"]
].head(10)

In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(
    books["description_length"].dropna(),
    vert=False
)
plt.title("Description length outlier check")
plt.xlabel("Words in description")
plt.show()

## Bivariate exploration and correlation

We examine whether books with longer descriptions also tend to have more rating activity.

This is exploratory. Correlation does not show causation.

In [ ]:
correlation_data = books[
    ["description_length", "ratings_count", "average_rating", "published_year"]
].dropna()

correlation_matrix = correlation_data.corr(numeric_only=True)
correlation_matrix

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    books["description_length"],
    books["ratings_count"],
    alpha=0.5
)
plt.title("Description length and ratings count")
plt.xlabel("Description length")
plt.ylabel("Ratings count")
plt.show()

## Probability example

We estimate the empirical probability that a randomly selected book from our collected catalogue has a usable description of at least 40 words.

This is a sample probability. It describes our collected dataset, not all books in the world.

In [ ]:
usable_description = books["description_length"] >= 40
probability_usable_description = usable_description.mean()

print(
    "Estimated probability of a description with at least 40 words:",
    round(probability_usable_description, 3)
)

## Normal distribution check

We first visualize average ratings with a histogram and a theoretical normal curve.

Then we use the Shapiro Wilk test on a sample when enough rating values are available.

Null hypothesis: average ratings are compatible with a normal distribution.

Alternative hypothesis: average ratings are not compatible with a normal distribution.

Alpha level: 0.05

In [ ]:
rating_values = books["average_rating"].dropna()

if len(rating_values) >= 3:
    sample_for_normality = rating_values.sample(
        min(len(rating_values), 500),
        random_state=RANDOM_STATE
    )

    shapiro_stat, shapiro_p = stats.shapiro(sample_for_normality)

    print("Test statistic:", round(shapiro_stat, 4))
    print("P value:", round(shapiro_p, 4))
    print("Alpha:", 0.05)

    if shapiro_p < 0.05:
        print("Decision: reject the null hypothesis")
    else:
        print("Decision: fail to reject the null hypothesis")
else:
    print("Not enough rating values for the normality test")

## Hypothesis test with critical values and rejection regions

We test whether books in the humans and nature collection and Yoga philosophy collection have different mean description lengths.

Null hypothesis: the two population means are equal.

Alternative hypothesis: the two population means are different.

We use a two sided independent t test with Welch correction because the two groups do not need to have equal variance.

Alpha level: 0.05.

The visualization marks the lower critical value, upper critical value, rejection regions, and observed test statistic.

In [ ]:
group_a = books.loc[
    books["source_category"] == "humans_and_nature",
    "description_length"
]
group_b = books.loc[
    books["source_category"] == "yoga_philosophy",
    "description_length"
]

group_a = group_a[group_a > 0]
group_b = group_b[group_b > 0]

alpha = 0.05

if len(group_a) >= 2 and len(group_b) >= 2:
    test_statistic, p_value = stats.ttest_ind(
        group_a,
        group_b,
        equal_var=False
    )

    variance_a = group_a.var(ddof=1)
    variance_b = group_b.var(ddof=1)
    n_a = len(group_a)
    n_b = len(group_b)

    numerator = (variance_a / n_a + variance_b / n_b) ** 2
    denominator = (
        (variance_a / n_a) ** 2 / (n_a - 1)
        + (variance_b / n_b) ** 2 / (n_b - 1)
    )
    degrees_freedom = numerator / denominator

    lcv = stats.t.ppf(alpha / 2, degrees_freedom)
    ucv = stats.t.ppf(1 - alpha / 2, degrees_freedom)

    print("Test statistic:", round(test_statistic, 4))
    print("P value:", round(p_value, 4))
    print("Alpha:", alpha)
    print("LCV:", round(lcv, 4))
    print("UCV:", round(ucv, 4))

    if p_value < alpha:
        print("Decision: reject the null hypothesis")
    else:
        print("Decision: fail to reject the null hypothesis")

    x = np.linspace(
        stats.t.ppf(0.001, degrees_freedom),
        stats.t.ppf(0.999, degrees_freedom),
        500
    )
    y = stats.t.pdf(x, degrees_freedom)

    plt.figure(figsize=(9, 5))
    plt.plot(x, y)

    left_x = x[x <= lcv]
    right_x = x[x >= ucv]

    plt.fill_between(
        left_x,
        stats.t.pdf(left_x, degrees_freedom),
        alpha=0.3
    )
    plt.fill_between(
        right_x,
        stats.t.pdf(right_x, degrees_freedom),
        alpha=0.3
    )

    plt.axvline(lcv, linestyle="--", label="LCV")
    plt.axvline(ucv, linestyle="--", label="UCV")
    plt.axvline(
        test_statistic,
        linestyle="-",
        label="Test statistic"
    )

    plt.title("Welch t test critical value visualization")
    plt.xlabel("t value")
    plt.ylabel("Probability density")
    plt.legend()
    plt.show()
else:
    print("Not enough observations in both groups for the t test")

## Correlation hypothesis test

We test whether description length and ratings count have a linear association.

Null hypothesis: population Pearson correlation equals zero.

Alternative hypothesis: population Pearson correlation is different from zero.

Alpha level: 0.05.

In [ ]:
corr_test_data = books[
    ["description_length", "ratings_count"]
].dropna()

if len(corr_test_data) >= 3:
    r_value, corr_p_value = stats.pearsonr(
        corr_test_data["description_length"],
        corr_test_data["ratings_count"]
    )

    n = len(corr_test_data)
    corr_test_statistic = (
        r_value * np.sqrt((n - 2) / (1 - r_value ** 2))
    )
    corr_df = n - 2

    corr_lcv = stats.t.ppf(0.025, corr_df)
    corr_ucv = stats.t.ppf(0.975, corr_df)

    print("Pearson correlation:", round(r_value, 4))
    print("Test statistic:", round(corr_test_statistic, 4))
    print("P value:", round(corr_p_value, 4))
    print("LCV:", round(corr_lcv, 4))
    print("UCV:", round(corr_ucv, 4))

    if corr_p_value < 0.05:
        print("Decision: reject the null hypothesis")
    else:
        print("Decision: fail to reject the null hypothesis")
else:
    print("Not enough complete observations for the correlation test")

# Day 3. NLP and feature engineering

## Purpose

Prepare one meaningful text field for each book.

The recommender combines title, subjects, description, and project theme. Repeating the project theme once gives the thematic collection some influence without overwhelming the actual book description.

### Expected result

A cleaned text column called `clean_text` and a TF IDF matrix called `tfidf_matrix`.

In [ ]:
books["model_text"] = (
    books["title"].fillna("") + " "
    + books["subjects"].fillna("") + " "
    + books["description"].fillna("") + " "
    + books["source_category"].fillna("")
)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

books["clean_text"] = books["model_text"].apply(clean_text)

books[
    ["title", "source_category", "clean_text"]
].head()

In [ ]:
model_books = books[
    books["clean_text"].str.len() > 40
].copy()

model_books = model_books.reset_index(drop=True)

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(
    model_books["clean_text"]
)

print("Books used for modeling:", len(model_books))
print("TF IDF matrix shape:", tfidf_matrix.shape)

# Day 4. K Means and PCA

## Purpose

Use K Means to explore whether books form interpretable textual groups.

We test several values of k with both inertia and silhouette score. We do not choose k from the elbow graph alone.

PCA reduces the TF IDF representation to two dimensions for visualization. PCA is used here for exploration and communication, not as proof that the clusters are objectively true categories.

In [ ]:
k_values = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    labels = model.fit_predict(tfidf_matrix)

    inertias.append(model.inertia_)
    silhouette_scores.append(
        silhouette_score(tfidf_matrix, labels)
    )

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.title("Elbow method")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), silhouette_scores, marker="o")
plt.title("Silhouette score by number of clusters")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.show()

In [ ]:
best_k = list(k_values)[int(np.argmax(silhouette_scores))]

kmeans = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=10
)

model_books["cluster"] = kmeans.fit_predict(
    tfidf_matrix
)

print("Selected k:", best_k)
model_books["cluster"].value_counts().sort_index()

## Interpret cluster vocabulary

The highest weighted TF IDF terms near each cluster center help us understand what each cluster may represent.

We describe clusters after seeing these terms. We do not assign spiritual or cultural meaning to a cluster only from the algorithm.

In [ ]:
terms = np.array(tfidf.get_feature_names_out())
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster_number in range(best_k):
    top_terms = terms[
        order_centroids[cluster_number, :12]
    ]
    print(
        "Cluster",
        cluster_number,
        ":",
        ", ".join(top_terms)
    )

In [ ]:
pca = PCA(
    n_components=2,
    random_state=RANDOM_STATE
)

tfidf_dense = tfidf_matrix.toarray()
pca_coordinates = pca.fit_transform(tfidf_dense)

model_books["pca_1"] = pca_coordinates[:, 0]
model_books["pca_2"] = pca_coordinates[:, 1]

plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    model_books["pca_1"],
    model_books["pca_2"],
    c=model_books["cluster"],
    alpha=0.7
)
plt.title("PCA view of book clusters")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.colorbar(scatter, label="Cluster")
plt.show()

print(
    "Variance represented by two PCA components:",
    round(pca.explained_variance_ratio_.sum(), 3)
)

# Day 5. Content based recommendation system

## Purpose

Recommend books whose text representation is similar to a selected title.

Cosine similarity measures the angle between TF IDF vectors. A higher value means the two books use more similar weighted vocabulary.

### Expected result

The function returns the closest books while excluding the selected book itself.

In [ ]:
similarity_matrix = cosine_similarity(tfidf_matrix)

def recommend_books(title, number_of_books=5):
    matching_rows = model_books[
        model_books["title"].str.lower() == title.lower()
    ]

    if matching_rows.empty:
        return pd.DataFrame({
            "message": ["Book title not found in the model dataset"]
        })

    book_index = matching_rows.index[0]

    similarity_scores = list(
        enumerate(similarity_matrix[book_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda item: item[1],
        reverse=True
    )

    recommended_indices = [
        index for index, score in similarity_scores
        if index != book_index
    ][:number_of_books]

    recommendations = model_books.loc[
        recommended_indices,
        [
            "title",
            "author",
            "source_category",
            "cluster"
        ]
    ].copy()

    recommendations["similarity_score"] = [
        similarity_matrix[book_index][index]
        for index in recommended_indices
    ]

    return recommendations.reset_index(drop=True)

In [ ]:
available_seed_titles = [
    title for title in SEED_TITLES
    if title.lower() in model_books["title"].str.lower().values
]

if available_seed_titles:
    display(
        recommend_books(
            available_seed_titles[0],
            number_of_books=5
        )
    )
else:
    print(
        "None of the seed titles were returned by the current API sample."
    )

## Natural language discovery

A reader can also describe what they want to read. We transform the request with the same TF IDF vectorizer and compare it with the books.

In [ ]:
def recommend_from_text(user_text, number_of_books=5):
    cleaned_request = clean_text(user_text)
    request_vector = tfidf.transform([cleaned_request])

    scores = cosine_similarity(
        request_vector,
        tfidf_matrix
    ).flatten()

    best_indices = scores.argsort()[::-1][:number_of_books]

    results = model_books.loc[
        best_indices,
        [
            "title",
            "author",
            "source_category",
            "cluster"
        ]
    ].copy()

    results["similarity_score"] = scores[best_indices]

    return results.reset_index(drop=True)

recommend_from_text(
    "I want a book about ecological connection, spirituality, and learning from the living world"
)

# Day 6. SQL database

## Purpose

Store the cleaned data in a relational structure that is easy to query.

The database uses four main entities.

Books contains one row per book.

Authors contains unique authors.

Subjects contains unique subject labels.

Book authors and book subjects connect the many to many relationships.

This avoids repeating the same author or subject information unnecessarily.

In [ ]:
model_books = model_books.reset_index(drop=True)
model_books["book_id"] = model_books.index + 1

books_table = model_books[
    [
        "book_id",
        "title",
        "description",
        "published_year",
        "average_rating",
        "ratings_count",
        "isbn",
        "language",
        "source",
        "source_category",
        "cluster"
    ]
].copy()

author_rows = []

for _, row in model_books.iterrows():
    authors = str(row["author"]).split(",")

    for author in authors:
        author = author.strip()

        if author and author.lower() != "nan":
            author_rows.append({
                "book_id": row["book_id"],
                "author_name": author
            })

book_authors_raw = pd.DataFrame(author_rows)

authors_table = (
    book_authors_raw[["author_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

authors_table["author_id"] = authors_table.index + 1

book_authors_table = book_authors_raw.merge(
    authors_table,
    on="author_name",
    how="left"
)[["book_id", "author_id"]]

In [ ]:
subject_rows = []

for _, row in model_books.iterrows():
    subjects = str(row["subjects"]).split(",")

    for subject in subjects:
        subject = subject.strip()

        if subject and subject.lower() != "nan":
            subject_rows.append({
                "book_id": row["book_id"],
                "subject_name": subject
            })

book_subjects_raw = pd.DataFrame(subject_rows)

subjects_table = (
    book_subjects_raw[["subject_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

subjects_table["subject_id"] = subjects_table.index + 1

book_subjects_table = book_subjects_raw.merge(
    subjects_table,
    on="subject_name",
    how="left"
)[["book_id", "subject_id"]]

In [ ]:
connection = sqlite3.connect(DATABASE_PATH)

books_table.to_sql(
    "books",
    connection,
    if_exists="replace",
    index=False
)

authors_table.to_sql(
    "authors",
    connection,
    if_exists="replace",
    index=False
)

subjects_table.to_sql(
    "subjects",
    connection,
    if_exists="replace",
    index=False
)

book_authors_table.to_sql(
    "book_authors",
    connection,
    if_exists="replace",
    index=False
)

book_subjects_table.to_sql(
    "book_subjects",
    connection,
    if_exists="replace",
    index=False
)

connection.close()

print("Database created at:", DATABASE_PATH)

## Beginner friendly SQL questions

These queries answer practical catalogue questions.

1. How many books are in each project theme?
2. Which authors occur most often?
3. What is the average rating by theme?
4. Which subjects appear most often?

In [ ]:
connection = sqlite3.connect(DATABASE_PATH)

query = '''
SELECT
    source_category,
    COUNT(*) AS number_of_books,
    ROUND(AVG(average_rating), 2) AS average_rating
FROM books
GROUP BY source_category
ORDER BY number_of_books DESC;
'''

sql_theme_summary = pd.read_sql_query(
    query,
    connection
)

connection.close()

sql_theme_summary

# Day 7. API enrichment

## Purpose

Use Google Books to improve missing descriptions, categories, ratings, publication information, and identifiers.

We only request enrichment for records that need it. This keeps the workflow understandable and reduces unnecessary API calls.

### Expected result

A final CSV called `books_enriched.csv`.

In [ ]:
books_enriched = model_books.copy()

books_needing_enrichment = books_enriched[
    books_enriched["description"].isna()
    | books_enriched["isbn"].isna()
    | books_enriched["average_rating"].isna()
].copy()

print(
    "Books that may benefit from enrichment:",
    len(books_needing_enrichment)
)

In [ ]:
for index, row in books_needing_enrichment.head(100).iterrows():
    query = f'intitle:"{row["title"]}"'

    if pd.notna(row["author"]):
        query += f' inauthor:"{row["author"]}"'

    params = {
        "q": query,
        "maxResults": 1,
        "printType": "books"
    }

    response = requests.get(
        "https://www.googleapis.com/books/v1/volumes",
        params=params,
        timeout=30
    )

    if response.status_code != 200:
        continue

    items = response.json().get("items", [])

    if not items:
        continue

    info = items[0].get("volumeInfo", {})

    if pd.isna(books_enriched.at[index, "description"]):
        books_enriched.at[index, "description"] = info.get(
            "description"
        )

    if pd.isna(books_enriched.at[index, "average_rating"]):
        books_enriched.at[index, "average_rating"] = info.get(
            "averageRating"
        )

    if pd.isna(books_enriched.at[index, "ratings_count"]):
        books_enriched.at[index, "ratings_count"] = info.get(
            "ratingsCount"
        )

    identifiers = info.get("industryIdentifiers", [])

    if pd.isna(books_enriched.at[index, "isbn"]) and identifiers:
        books_enriched.at[index, "isbn"] = identifiers[0].get(
            "identifier"
        )

    time.sleep(1)

books_enriched.to_csv(
    PROCESSED_DATA / "books_enriched.csv",
    index=False
)

books_enriched.head()

# Day 7 checkpoint

At this point the project contains a complete first version of the analytical pipeline.

Data collection uses one scraping source and two book APIs.

Cleaning produces a consistent book table.

EDA checks missing values, distributions, outliers, probability, correlation, normality, and formal hypothesis testing.

NLP transforms book text with TF IDF.

K Means explores textual clusters.

PCA provides a two dimensional view of the clusters.

Cosine similarity powers title based and natural language recommendations.

SQL stores books, authors, and subjects in relational tables.

API enrichment improves incomplete records.

The next stage is Day 8. We should inspect recommendation quality before adding more filters. The early Streamlit prototype included in the repository can already load the processed data and provide a simple thematic browser.